In [1]:
import pandas as pd
import searoute as sr

In [2]:
from _helpers import mock_snakemake
snakemake = mock_snakemake(
    "prepare_chokepoints",
)

### File paths

In [3]:
trade_options_fn = snakemake.input.trade_options
bus_locations_fn = snakemake.input.bus_locations

# Output
trade_options_chokepoints_fn = snakemake.output.trade_options_chokepoints

### Load data

In [4]:
trade_options = pd.read_csv(trade_options_fn, header=0)
bus_locations = pd.read_csv(bus_locations_fn, header=0)

print(f"Trade options: {len(trade_options)} routes")
print(f"Bus locations: {len(bus_locations)} regions")

# Build region -> (lon, lat) mapping for searoute
region_coords = {
    row["region_name"]: [row["long"], row["lat"]]
    for _, row in bus_locations.iterrows()
}

print(f"\nRegion coordinates:")
for name, coords in region_coords.items():
    print(f"  {name}: {coords}")

Trade options: 210 routes
Bus locations: 15 regions

Region coordinates:
  Europe: [19.649327, 48.565099]
  Far_West_Europe: [-6.031871, 45.859412]
  Middle_East: [49.90389, 26.470573]
  North_West_Africa: [-1.10671, 18.020528]
  Subsaharan_Africa: [26.802531, -10.790146]
  North_America: [-102.658843, 38.856828]
  Eurasia: [70.542647, 61.122017]
  South_America: [-59.035985, -10.214014]
  South_South_America: [-69.003577, -39.695915]
  Central_America: [-91.401331, 15.890905]
  West_Asia: [77.695851, 21.221111]
  East_Asia: [112.17198, 33.076197]
  Pacific_Asia: [121.318708, -2.884526]
  East_East_Asia: [139.377626, 35.782171]
  Oceania: [146.765368, -28.574874]


### Compute chokepoints per route using `searoute`

For each shipping route in `trade_opt.csv`, compute the shortest sea route and record which passages (chokepoints) are traversed.

In [5]:
chokepoints_list = []

for idx, row in trade_options.iterrows():
    r_from = row["region_from"]
    r_to = row["region_to"]

    # Only compute for shipping routes
    if row["shipping"] != 1:
        chokepoints_list.append("")
        continue

    origin = region_coords.get(r_from)
    destination = region_coords.get(r_to)

    if origin is None or destination is None:
        print(f"  Warning: missing coordinates for {r_from} or {r_to}")
        chokepoints_list.append("")
        continue

    try:
        route = sr.searoute(
            origin, destination,
            restrictions=[],  # allow all passages so we can detect them
            return_passages=True,
        )
        passages = route["properties"].get("traversed_passages", [])
        chokepoints_list.append(";".join(passages) if passages else "")
    except Exception as e:
        print(f"  Error computing route {r_from} -> {r_to}: {e}")
        chokepoints_list.append("")

trade_options["chokepoints"] = chokepoints_list

print(f"\nRoutes with chokepoints: {(trade_options['chokepoints'] != '').sum()} / {len(trade_options)}")


Routes with chokepoints: 150 / 210


In [6]:
# Display routes and their chokepoints
display_cols = ["region_from", "region_to", "chokepoints"]
routes_with_cp = trade_options[trade_options["chokepoints"] != ""][display_cols]
print(f"Routes passing through chokepoints ({len(routes_with_cp)}):")
routes_with_cp

Routes passing through chokepoints (150):


,region_from,region_to,chokepoints
0,North_America,East_Asia,panama
1,East_Asia,North_America,panama
6,Central_America,West_Asia,babalmandab;gibraltar;panama;suez
7,West_Asia,Central_America,gibraltar;suez;panama;babalmandab
8,Eurasia,South_South_America,panama
...,...,...,...
205,North_West_Africa,Middle_East,south_africa;ormuz
206,Europe,North_West_Africa,gibraltar
207,North_West_Africa,Europe,gibraltar
208,Eurasia,East_East_Asia,bering;northwest


### Save enriched trade options

In [7]:
trade_options.to_csv(trade_options_chokepoints_fn, index=False)
print(f"Saved to {trade_options_chokepoints_fn}")

Saved to /mnt/c/Users/scl38887/Documents/git/shift/resources/trade_opt_chokepoints.csv
